In [1]:
pip install pandas numpy scikit-learn xgboost imbalanced-learn shap matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("Maternal_Risk.csv")  # update filename if yours differs

print(df.shape)
print(df.columns.tolist())
print(df.head())
print(df.dtypes)
print(df.isnull().sum())

(6103, 11)
['Patient ID', 'Name', 'Age', 'Body Temperature(F) ', 'Heart rate(bpm)', 'Systolic Blood Pressure(mm Hg)', 'Diastolic Blood Pressure(mm Hg)', 'BMI(kg/m 2)', 'Blood Glucose(HbA1c)', 'Blood Glucose(Fasting hour-mg/dl)', 'Status']
   Patient ID     Name  Age  Body Temperature(F)   Heart rate(bpm)  \
0         101  Bhavana   20                  97.5               91   
1         102  Mithila   45                  97.7               99   
2         103  Phoolan   29                  98.6               84   
3         104  Joyonti   26                  99.5              135   
4         105     Vana   38                 102.5               51   

   Systolic Blood Pressure(mm Hg)  Diastolic Blood Pressure(mm Hg)  \
0                             161                              100   
1                              99                               94   
2                             129                               87   
3                             161                           

In [4]:
print(df['Status'].value_counts())
print(df['Status'].unique())
print(df.duplicated().sum())
print(df.duplicated(subset=[c for c in df.columns if c not in ['Patient ID', 'Name']]).sum())

Status
high risk    2059
mid risk     2043
low risk     2001
Name: count, dtype: int64
<ArrowStringArray>
['high risk', 'mid risk', 'low risk']
Length: 3, dtype: str
0
1


In [5]:
feature_and_target_cols = [c for c in df.columns if c not in ['Patient ID', 'Name']]
dupes = df[df.duplicated(subset=feature_and_target_cols, keep=False)]
print(dupes)

      Patient ID      Name  Age  Body Temperature(F)   Heart rate(bpm)  \
3442        3543     Vatya   24                  98.6               75   
3537        3638  Abhijita   24                  98.6               75   

      Systolic Blood Pressure(mm Hg)  Diastolic Blood Pressure(mm Hg)  \
3442                             126                               90   
3537                             126                               90   

      BMI(kg/m 2)  Blood Glucose(HbA1c)  Blood Glucose(Fasting hour-mg/dl)  \
3442         19.0                    37                                5.9   
3537         19.0                    37                                5.9   

        Status  
3442  low risk  
3537  low risk  


In [6]:
# --- No rows dropped — confirmed as coincidental match between two distinct patients, not a duplicate entry ---
print(df.shape)   # should stay (6103, 11), unchanged

(6103, 11)


In [7]:
feature_cols_raw = ['Age', 'Body Temperature(F) ', 'Heart rate(bpm)',
                     'Systolic Blood Pressure(mm Hg)', 'Diastolic Blood Pressure(mm Hg)',
                     'BMI(kg/m 2)', 'Blood Glucose(HbA1c)', 'Blood Glucose(Fasting hour-mg/dl)']

print(df[feature_cols_raw].describe())

               Age  Body Temperature(F)   Heart rate(bpm)  \
count  6103.000000           6103.000000      6103.000000   
mean     26.425037             98.665574        86.100770   
std       6.390205              1.590983        22.627587   
min      15.000000             39.600000        45.000000   
25%      22.000000             98.600000        72.000000   
50%      25.000000             98.600000        80.000000   
75%      30.000000             98.800000        91.000000   
max     250.000000            104.000000       150.000000   

       Systolic Blood Pressure(mm Hg)  Diastolic Blood Pressure(mm Hg)  \
count                     6103.000000                      6103.000000   
mean                       129.218253                        87.257578   
std                         17.234217                         7.793099   
min                         90.000000                         9.000000   
25%                        120.000000                        82.000000   
50%   

In [8]:
before = df.shape[0]
print(f"Before cleaning: {before}")

# --- Apply paper's exact validated clinical ranges ---
df_clean = df[
    (df['Age'] >= 15) & (df['Age'] <= 48) &
    (df['Body Temperature(F) '] >= 95.0) & (df['Body Temperature(F) '] <= 104.0) &
    (df['Diastolic Blood Pressure(mm Hg)'] >= 50)
].copy()

after = df_clean.shape[0]
print(f"After cleaning: {after}")
print(f"Removed: {before - after} (paper reports 45)")

# --- Rename mislabeled unit columns for clarity going forward ---
df_clean = df_clean.rename(columns={
    'Blood Glucose(HbA1c)': 'Blood Glucose(HbA1c mmol/mol)',
    'Blood Glucose(Fasting hour-mg/dl)': 'Blood Glucose(Fasting mmol/L)'
})

print(df_clean.columns.tolist())
print(df_clean[['Age', 'Body Temperature(F) ', 'Diastolic Blood Pressure(mm Hg)']].describe())

Before cleaning: 6103
After cleaning: 6058
Removed: 45 (paper reports 45)
['Patient ID', 'Name', 'Age', 'Body Temperature(F) ', 'Heart rate(bpm)', 'Systolic Blood Pressure(mm Hg)', 'Diastolic Blood Pressure(mm Hg)', 'BMI(kg/m 2)', 'Blood Glucose(HbA1c mmol/mol)', 'Blood Glucose(Fasting mmol/L)', 'Status']
               Age  Body Temperature(F)   Diastolic Blood Pressure(mm Hg)
count  6058.000000           6058.000000                      6058.000000
mean     26.382800             98.705381                        87.325190
std       5.669496              1.356713                         7.641088
min      15.000000             95.000000                        57.000000
25%      22.000000             98.600000                        82.000000
50%      25.000000             98.600000                        87.000000
75%      30.000000             98.800000                        92.000000
max      48.000000            104.000000                       142.000000


In [9]:
print(df_clean.isnull().sum().sum())
print(df_clean.duplicated().sum())
print(df_clean.duplicated(subset=[c for c in df_clean.columns if c not in ['Patient ID', 'Name']]).sum())
print(df_clean.shape)

0
0
1
(6058, 11)


In [ ]:
# --- Save the cleaned dataset (6,058 rows, identifiers dropped) for backend/scripts/compute_feature_ranges.py ---
df_clean.drop(columns=['Patient ID', 'Name']).to_csv("maternal_clean.csv", index=False)

In [10]:
# --- Define X (exclude identifiers) and y (encode target, matching paper's convention) ---
identifier_cols = ['Patient ID', 'Name']
target_col = 'Status'

feature_cols_maternal = [c for c in df_clean.columns if c not in identifier_cols + [target_col]]

X = df_clean[feature_cols_maternal].copy()

status_mapping = {'low risk': 0, 'mid risk': 1, 'high risk': 2}
y = df_clean[target_col].map(status_mapping)

# --- Sanity checks before moving forward ---
print(feature_cols_maternal)
print(X.shape)
print(y.shape)
print(X.dtypes)
print(y.value_counts())
print(y.isnull().sum())   # must be 0 — a non-zero count would mean status_mapping missed a label

['Age', 'Body Temperature(F) ', 'Heart rate(bpm)', 'Systolic Blood Pressure(mm Hg)', 'Diastolic Blood Pressure(mm Hg)', 'BMI(kg/m 2)', 'Blood Glucose(HbA1c mmol/mol)', 'Blood Glucose(Fasting mmol/L)']
(6058, 8)
(6058,)
Age                                  int64
Body Temperature(F)                float64
Heart rate(bpm)                      int64
Systolic Blood Pressure(mm Hg)       int64
Diastolic Blood Pressure(mm Hg)      int64
BMI(kg/m 2)                        float64
Blood Glucose(HbA1c mmol/mol)        int64
Blood Glucose(Fasting mmol/L)      float64
dtype: object
Status
1    2043
2    2016
0    1999
Name: count, dtype: int64
0


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(4846, 8) (1212, 8)
Status
1    0.337185
2    0.332852
0    0.329963
Name: proportion, dtype: float64
Status
1    0.337459
2    0.332508
0    0.330033
Name: proportion, dtype: float64


In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Sanity checks ---
print(X_train_scaled.mean(axis=0).round(3))   # should be ~0 for every feature
print(X_train_scaled.std(axis=0).round(3))    # should be ~1 for every feature
print(X_test_scaled.mean(axis=0).round(3))    # will NOT be exactly 0 — expected, since scaler was fit on train only

[-0. -0. -0.  0.  0.  0. -0.  0.]
[1. 1. 1. 1. 1. 1. 1. 1.]
[-0.035 -0.014 -0.042 -0.014  0.059 -0.01  -0.025  0.046]


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score

results_maternal = {}

# 1. Logistic Regression (scaled input)
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
results_maternal['Logistic Regression'] = lr.predict(X_test_scaled)

# 2. Random Forest (raw input, no scaling needed)
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
results_maternal['Random Forest'] = rf.predict(X_test)

# 3. SVM (scaled input)
svm = SVC(random_state=42)
svm.fit(X_train_scaled, y_train)
results_maternal['SVM'] = svm.predict(X_test_scaled)

# 4. XGBoost (raw input; y already 0/1/2, no shift needed this time)
xgb = XGBClassifier(random_state=42, eval_metric='mlogloss')
xgb.fit(X_train, y_train)
results_maternal['XGBoost'] = xgb.predict(X_test)

# --- Evaluate all 4 ---
class_names_maternal = ['Low', 'Mid', 'High']
for name, preds in results_maternal.items():
    print(f"\n===== {name} =====")
    print(classification_report(y_test, preds, target_names=class_names_maternal))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, preds))
    print(f"Macro F1: {f1_score(y_test, preds, average='macro'):.4f}")


===== Logistic Regression =====
              precision    recall  f1-score   support

         Low       0.58      0.73      0.65       400
         Mid       0.52      0.52      0.52       409
        High       0.61      0.45      0.52       403

    accuracy                           0.57      1212
   macro avg       0.57      0.57      0.56      1212
weighted avg       0.57      0.57      0.56      1212

Confusion Matrix:
[[294  83  23]
 [105 212  92]
 [106 115 182]]
Macro F1: 0.5625

===== Random Forest =====
              precision    recall  f1-score   support

         Low       0.99      1.00      0.99       400
         Mid       0.99      0.99      0.99       409
        High       0.99      0.99      0.99       403

    accuracy                           0.99      1212
   macro avg       0.99      0.99      0.99      1212
weighted avg       0.99      0.99      0.99      1212

Confusion Matrix:
[[399   0   1]
 [  4 404   1]
 [  0   5 398]]
Macro F1: 0.9909

===== SVM =====

In [14]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_cv = RandomForestClassifier(random_state=42)
rf_scores = cross_val_score(rf_cv, X, y, cv=skf, scoring='f1_macro')

xgb_cv = XGBClassifier(random_state=42, eval_metric='mlogloss')
xgb_scores = cross_val_score(xgb_cv, X, y, cv=skf, scoring='f1_macro')

print("Random Forest CV macro F1 per fold:", rf_scores)
print(f"Random Forest mean: {rf_scores.mean():.4f} (+/- {rf_scores.std():.4f})")
print()
print("XGBoost CV macro F1 per fold:", xgb_scores)
print(f"XGBoost mean: {xgb_scores.mean():.4f} (+/- {xgb_scores.std():.4f})")

Random Forest CV macro F1 per fold: [0.99260333 0.99176375 0.99424595 0.98683225 0.99506125]
Random Forest mean: 0.9921 (+/- 0.0029)

XGBoost CV macro F1 per fold: [0.99094714 0.99176212 0.99424419 0.98847342 0.99423722]
XGBoost mean: 0.9919 (+/- 0.0022)


In [15]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from xgboost import XGBClassifier

# --- Param search space (same structure as fetal, no sample_weight needed here) ---
param_dist_maternal = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2]
}

xgb_base_maternal = XGBClassifier(random_state=42, eval_metric='mlogloss')

skf_maternal = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search_maternal = RandomizedSearchCV(
    estimator=xgb_base_maternal,
    param_distributions=param_dist_maternal,
    n_iter=50,
    scoring='f1_macro',
    cv=skf_maternal,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# --- No sample_weight here: classes are already balanced, unlike fetal ---
search_maternal.fit(X_train, y_train)

print("Best params:", search_maternal.best_params_)
print("Best CV macro F1:", search_maternal.best_score_)

# --- Evaluate the tuned model on the held-out test set ---
best_xgb_maternal = search_maternal.best_estimator_
test_preds_maternal = best_xgb_maternal.predict(X_test)   # no +1 shift needed — y is already 0/1/2

class_names_maternal = ['Low', 'Mid', 'High']
print("\n===== Tuned XGBoost — Test Set Performance (Maternal) =====")
print(classification_report(y_test, test_preds_maternal, target_names=class_names_maternal))
print("Confusion Matrix:")
print(confusion_matrix(y_test, test_preds_maternal))
print(f"Macro F1: {f1_score(y_test, test_preds_maternal, average='macro'):.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'subsample': 1.0, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 8, 'learning_rate': 0.2, 'gamma': 0.1, 'colsample_bytree': 0.7}
Best CV macro F1: 0.9923822887620528

===== Tuned XGBoost — Test Set Performance (Maternal) =====
              precision    recall  f1-score   support

         Low       0.99      1.00      0.99       400
         Mid       0.99      0.99      0.99       409
        High       1.00      0.99      0.99       403

    accuracy                           0.99      1212
   macro avg       0.99      0.99      0.99      1212
weighted avg       0.99      0.99      0.99      1212

Confusion Matrix:
[[399   0   1]
 [  4 405   0]
 [  0   6 397]]
Macro F1: 0.9909


In [16]:
import shap
import matplotlib.pyplot as plt
import numpy as np

# --- TreeExplainer for tuned maternal XGBoost ---
explainer_maternal = shap.TreeExplainer(best_xgb_maternal)
shap_values_maternal = explainer_maternal.shap_values(X_test)

print(type(shap_values_maternal))
if isinstance(shap_values_maternal, list):
    print(f"List of {len(shap_values_maternal)} arrays, each shape: {shap_values_maternal[0].shape}")
else:
    print(f"Array shape: {shap_values_maternal.shape}")

class_names_maternal = ['Low', 'Mid', 'High']

# --- Handle both possible SHAP return formats safely, same caution as fetal ---
def get_class_shap(shap_values, class_idx):
    if isinstance(shap_values, list):
        return shap_values[class_idx]
    else:
        return shap_values[:, :, class_idx]

# --- Summary plots per class, saved to file ---
for i, cname in enumerate(class_names_maternal):
    class_shap = get_class_shap(shap_values_maternal, i)
    shap.summary_plot(class_shap, X_test, feature_names=feature_cols_maternal, show=False)
    plt.title(f"SHAP Summary — {cname} Risk (Maternal)")
    plt.tight_layout()
    plt.savefig(f"shap_summary_maternal_{cname.lower()}.png", dpi=150, bbox_inches='tight')
    plt.close()

print("Saved: shap_summary_maternal_low.png, shap_summary_maternal_mid.png, shap_summary_maternal_high.png")

<class 'numpy.ndarray'>
Array shape: (1212, 8, 3)
Saved: shap_summary_maternal_low.png, shap_summary_maternal_mid.png, shap_summary_maternal_high.png


In [17]:
# --- Single-prediction waterfall example: pick a true High-risk case ---
high_idx = np.where(y_test.values == 2)[0][0]   # first true High-risk case in test set
sample = X_test.iloc[[high_idx]]

pred_class = best_xgb_maternal.predict(sample)[0]
print(f"True label: High | Predicted: {class_names_maternal[pred_class]}")

class_shap_sample = get_class_shap(shap_values_maternal, pred_class)

explanation_maternal = shap.Explanation(
    values=class_shap_sample[high_idx],
    base_values=explainer_maternal.expected_value[pred_class],
    data=sample.values[0],
    feature_names=feature_cols_maternal
)

shap.plots.waterfall(explanation_maternal, show=False)
plt.tight_layout()
plt.savefig("shap_waterfall_maternal_high_example.png", dpi=150, bbox_inches='tight')
plt.close()

print("Saved: shap_waterfall_maternal_high_example.png")

True label: High | Predicted: High
Saved: shap_waterfall_maternal_high_example.png


In [18]:
from sklearn.metrics import f1_score, classification_report

# --- Predictions on TRAIN set ---
train_preds_maternal = best_xgb_maternal.predict(X_train)   # no shift — y already 0/1/2

print("===== Tuned XGBoost — TRAIN Set Performance (Maternal) =====")
print(classification_report(y_train, train_preds_maternal, target_names=class_names_maternal))
print(f"Train Macro F1: {f1_score(y_train, train_preds_maternal, average='macro'):.4f}")

print("\n===== Tuned XGBoost — TEST Set Performance (for comparison) =====")
test_preds_maternal = best_xgb_maternal.predict(X_test)
print(classification_report(y_test, test_preds_maternal, target_names=class_names_maternal))
print(f"Test Macro F1: {f1_score(y_test, test_preds_maternal, average='macro'):.4f}")

train_f1_maternal = f1_score(y_train, train_preds_maternal, average='macro')
test_f1_maternal = f1_score(y_test, test_preds_maternal, average='macro')
print(f"\nGap (Train F1 - Test F1): {train_f1_maternal - test_f1_maternal:.4f}")

===== Tuned XGBoost — TRAIN Set Performance (Maternal) =====
              precision    recall  f1-score   support

         Low       1.00      1.00      1.00      1599
         Mid       1.00      1.00      1.00      1634
        High       1.00      1.00      1.00      1613

    accuracy                           1.00      4846
   macro avg       1.00      1.00      1.00      4846
weighted avg       1.00      1.00      1.00      4846

Train Macro F1: 0.9992

===== Tuned XGBoost — TEST Set Performance (for comparison) =====
              precision    recall  f1-score   support

         Low       0.99      1.00      0.99       400
         Mid       0.99      0.99      0.99       409
        High       1.00      0.99      0.99       403

    accuracy                           0.99      1212
   macro avg       0.99      0.99      0.99      1212
weighted avg       0.99      0.99      0.99      1212

Test Macro F1: 0.9909

Gap (Train F1 - Test F1): 0.0082


In [19]:
# --- Real median profile per class ---
class_profiles_maternal = df_clean.groupby('Status')[feature_cols_maternal].median()
print(class_profiles_maternal.T)

Status                           high risk  low risk  mid risk
Age                                   26.0      24.0      27.0
Body Temperature(F)                   98.7      98.6      98.6
Heart rate(bpm)                       91.0      78.0      82.0
Systolic Blood Pressure(mm Hg)       126.0     127.0     129.0
Diastolic Blood Pressure(mm Hg)       88.0      84.0      88.0
BMI(kg/m 2)                           21.6      20.9      21.6
Blood Glucose(HbA1c mmol/mol)         38.0      37.0      38.0
Blood Glucose(Fasting mmol/L)          5.8       5.2       5.8


In [20]:
test_cases_maternal = class_profiles_maternal[feature_cols_maternal]
status_labels = ['high risk', 'low risk', 'mid risk']
status_to_encoded = {'low risk': 0, 'mid risk': 1, 'high risk': 2}

for status_label in status_labels:
    sample = test_cases_maternal.loc[[status_label]]
    true_encoded = status_to_encoded[status_label]
    pred = best_xgb_maternal.predict(sample)[0]
    proba = best_xgb_maternal.predict_proba(sample)[0]
    print(f"True class: {status_label} | Predicted: {class_names_maternal[pred]} | Probabilities: {dict(zip(class_names_maternal, proba.round(3)))}")

True class: high risk | Predicted: Mid | Probabilities: {'Low': np.float32(0.019), 'Mid': np.float32(0.936), 'High': np.float32(0.045)}
True class: low risk | Predicted: Low | Probabilities: {'Low': np.float32(0.998), 'Mid': np.float32(0.001), 'High': np.float32(0.001)}
True class: mid risk | Predicted: Low | Probabilities: {'Low': np.float32(0.992), 'Mid': np.float32(0.006), 'High': np.float32(0.001)}


In [21]:
# --- Pick one real, representative row per class from the test set (closest to that class's median profile) ---
import numpy as np

for status_label, encoded_val in [('low risk', 0), ('mid risk', 1), ('high risk', 2)]:
    class_mask = (y_test == encoded_val).values
    X_test_class = X_test[class_mask]

    # Find the real row closest to the class median (Euclidean distance, on scaled features to be fair across units)
    class_median = test_cases_maternal.loc[[status_label]].values[0]
    class_median_scaled = scaler.transform(test_cases_maternal.loc[[status_label]])[0]
    X_test_class_scaled = scaler.transform(X_test_class)

    distances = np.linalg.norm(X_test_class_scaled - class_median_scaled, axis=1)
    closest_idx = distances.argmin()
    closest_real_row = X_test_class.iloc[[closest_idx]]

    pred = best_xgb_maternal.predict(closest_real_row)[0]
    proba = best_xgb_maternal.predict_proba(closest_real_row)[0]
    print(f"True: {status_label} | Closest real row predicted: {class_names_maternal[pred]} | Probabilities: {dict(zip(class_names_maternal, proba.round(3)))}")
    print(closest_real_row.to_dict('records')[0])
    print()

True: low risk | Closest real row predicted: Low | Probabilities: {'Low': np.float32(0.998), 'Mid': np.float32(0.001), 'High': np.float32(0.001)}
{'Age': 24, 'Body Temperature(F) ': 98.6, 'Heart rate(bpm)': 74, 'Systolic Blood Pressure(mm Hg)': 128, 'Diastolic Blood Pressure(mm Hg)': 85, 'BMI(kg/m 2)': 21.1, 'Blood Glucose(HbA1c mmol/mol)': 38, 'Blood Glucose(Fasting mmol/L)': 5.0}

True: mid risk | Closest real row predicted: Mid | Probabilities: {'Low': np.float32(0.004), 'Mid': np.float32(0.981), 'High': np.float32(0.015)}
{'Age': 31, 'Body Temperature(F) ': 98.8, 'Heart rate(bpm)': 89, 'Systolic Blood Pressure(mm Hg)': 132, 'Diastolic Blood Pressure(mm Hg)': 86, 'BMI(kg/m 2)': 20.5, 'Blood Glucose(HbA1c mmol/mol)': 37, 'Blood Glucose(Fasting mmol/L)': 5.6}

True: high risk | Closest real row predicted: Mid | Probabilities: {'Low': np.float32(0.001), 'Mid': np.float32(0.99), 'High': np.float32(0.008)}
{'Age': 22, 'Body Temperature(F) ': 98.6, 'Heart rate(bpm)': 76, 'Systolic Blood P

In [22]:
results_df_maternal = X_test.copy()
results_df_maternal['True_Status'] = y_test.values
results_df_maternal['Predicted_Status'] = test_preds_maternal
results_df_maternal['True_Label'] = results_df_maternal['True_Status'].map({0: 'Low', 1: 'Mid', 2: 'High'})
results_df_maternal['Predicted_Label'] = results_df_maternal['Predicted_Status'].map({0: 'Low', 1: 'Mid', 2: 'High'})

errors_maternal = results_df_maternal[results_df_maternal['True_Status'] != results_df_maternal['Predicted_Status']]

print(f"Total misclassified: {len(errors_maternal)} out of {len(results_df_maternal)}")
print(errors_maternal[['True_Label', 'Predicted_Label'] + feature_cols_maternal])

Total misclassified: 11 out of 1212
     True_Label Predicted_Label  Age  Body Temperature(F)   Heart rate(bpm)  \
5241        Low            High   26                  98.6               77   
264         Mid             Low   20                  98.6               86   
3562       High             Mid   22                  98.6               76   
2699       High             Mid   21                  98.6              135   
1362       High             Mid   22                  98.8               70   
1811        Mid             Low   26                  98.6               77   
3904        Mid             Low   22                  97.6               75   
587        High             Mid   22                  98.6               98   
5085       High             Mid   22                  98.6               94   
13         High             Mid   19                  98.6               84   
1677        Mid             Low   23                  98.6               82   

      Systolic 

In [23]:
train_sizes_frac_maternal = np.linspace(0.1, 1.0, 8)
train_f1_scores_maternal = []
val_f1_scores_maternal = []

skf_lc_maternal = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for frac in train_sizes_frac_maternal:
    fold_train_f1 = []
    fold_val_f1 = []

    for train_idx, val_idx in skf_lc_maternal.split(X, y):
        X_tr_full, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr_full, y_val = y.iloc[train_idx], y.iloc[val_idx]

        if frac >= 1.0:
            X_tr_sub, y_tr_sub = X_tr_full, y_tr_full
        else:
            n_samples = int(len(X_tr_full) * frac)
            X_tr_sub, _, y_tr_sub, _ = train_test_split(
                X_tr_full, y_tr_full,
                train_size=n_samples,
                stratify=y_tr_full,
                random_state=42
            )

        # No sample_weight here — classes are balanced
        model = XGBClassifier(**search_maternal.best_params_, random_state=42, eval_metric='mlogloss')
        model.fit(X_tr_sub, y_tr_sub)

        train_preds = model.predict(X_tr_sub)
        val_preds = model.predict(X_val)

        fold_train_f1.append(f1_score(y_tr_sub, train_preds, average='macro'))
        fold_val_f1.append(f1_score(y_val, val_preds, average='macro'))

    train_f1_scores_maternal.append(np.mean(fold_train_f1))
    val_f1_scores_maternal.append(np.mean(fold_val_f1))

actual_sizes_maternal = (train_sizes_frac_maternal * len(X) * 0.8).astype(int)

plt.figure(figsize=(8,5))
plt.plot(actual_sizes_maternal, train_f1_scores_maternal, 'o-', label='Train F1 (macro)')
plt.plot(actual_sizes_maternal, val_f1_scores_maternal, 'o-', label='Validation F1 (macro)')
plt.xlabel('Training set size (approx.)')
plt.ylabel('Macro F1')
plt.title('Learning Curve — XGBoost (Maternal Risk Model)')
plt.legend()
plt.grid(True)
plt.savefig('learning_curve_maternal.png', dpi=150, bbox_inches='tight')
plt.close()

for size, tr, val in zip(actual_sizes_maternal, train_f1_scores_maternal, val_f1_scores_maternal):
    print(f"Train size: {size:4d} | Train F1: {tr:.4f} | Val F1: {val:.4f}")

print("\nSaved: learning_curve_maternal.png")

Train size:  484 | Train F1: 0.9992 | Val F1: 0.9523
Train size: 1107 | Train F1: 0.9987 | Val F1: 0.9767
Train size: 1730 | Train F1: 0.9987 | Val F1: 0.9852
Train size: 2353 | Train F1: 0.9987 | Val F1: 0.9890
Train size: 2977 | Train F1: 0.9989 | Val F1: 0.9914
Train size: 3600 | Train F1: 0.9986 | Val F1: 0.9918
Train size: 4223 | Train F1: 0.9984 | Val F1: 0.9911
Train size: 4846 | Train F1: 0.9986 | Val F1: 0.9916

Saved: learning_curve_maternal.png


In [24]:
import joblib
import json
from datetime import datetime

# --- Save the trained model ---
joblib.dump(best_xgb_maternal, 'maternal_risk_xgboost_model.joblib')

# --- Save feature column order (critical: backend must match this exact order) ---
with open('maternal_risk_feature_order.json', 'w') as f:
    json.dump(feature_cols_maternal, f, indent=2)

# --- Save label mapping (matches paper's convention: Low=0, Mid=1, High=2 — no shift needed) ---
label_mapping_maternal = {
    "model_output_to_status": {0: "low risk", 1: "mid risk", 2: "high risk"},
    "status_to_label": {"low risk": "Low", "mid risk": "Mid", "high risk": "High"}
}
with open('maternal_risk_label_mapping.json', 'w') as f:
    json.dump(label_mapping_maternal, f, indent=2)

# --- Save metadata for reference ---
metadata_maternal = {
    "model_type": "XGBoost",
    "task": "Maternal Risk Classification (Low/Mid/High)",
    "trained_on": datetime.now().strftime("%Y-%m-%d"),
    "n_features": len(feature_cols_maternal),
    "best_params": search_maternal.best_params_,
    "test_macro_f1": 0.9909,
    "test_accuracy": 0.99,
    "cv_macro_f1": 0.9924,
    "requires_scaling": False,
    "unit_notes": {
        "Blood Glucose(HbA1c mmol/mol)": "mmol/mol (IFCC), NOT percent",
        "Blood Glucose(Fasting mmol/L)": "mmol/L, NOT mg/dl despite original source column name"
    },
    "notes": "No class_weight/SMOTE used — classes are near-perfectly balanced (~33% each). Feature order in maternal_risk_feature_order.json must be preserved exactly at inference time. Trained on cleaned data (6058 rows, 45 physiologically implausible raw rows removed per Hossain et al. 2026 validated ranges)."
}
with open('maternal_risk_model_metadata.json', 'w') as f:
    json.dump(metadata_maternal, f, indent=2)

print("Saved files:")
print("- maternal_risk_xgboost_model.joblib")
print("- maternal_risk_feature_order.json")
print("- maternal_risk_label_mapping.json")
print("- maternal_risk_model_metadata.json")

Saved files:
- maternal_risk_xgboost_model.joblib
- maternal_risk_feature_order.json
- maternal_risk_label_mapping.json
- maternal_risk_model_metadata.json


In [25]:
# --- Load-and-verify ---
loaded_model_maternal = joblib.load('maternal_risk_xgboost_model.joblib')

loaded_preds_maternal = loaded_model_maternal.predict(X_test)
original_preds_maternal = best_xgb_maternal.predict(X_test)

print((loaded_preds_maternal == original_preds_maternal).all())   # must be True

from sklearn.metrics import f1_score
print(f1_score(y_test, loaded_preds_maternal, average='macro'))   # should match 0.9909

True
0.9909463839439026
